In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_absolute_error, root_mean_squared_error
import xgboost as xgb
import optuna

optuna.logging.set_verbosity(optuna.logging.WARNING)

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 5)

### Loading data

In [ ]:
df = pd.read_csv("../data/processed/features.csv")
print(f"Shape: {df.shape}")
print(f"Train rows: {(~df['is_test']).sum()}")
print(f"Test rows:  {df['is_test'].sum()}")

### Train validation split

In [ ]:
feature_cols = [
    "cumulative_energy_MJ", "time_since_heating_s",
    *[f"Bulk_{i}_cum" for i in range(1, 16)],
    "total_bulk_cum",
    *[f"Wire_{i}_cum" for i in range(1, 10)],
    "total_wire_cum",
    "gas_total", "prev_temp", "measurement_index",
]

train_df = df[~df["is_test"]].copy()
test_df = df[df["is_test"]].copy()

X_train = train_df[feature_cols]
y_train = train_df["Temperature"]
groups = train_df["key"]

print(f"X_train: {X_train.shape}")
print(f"Unique melts in train: {groups.nunique()}")
print(f"prev_temp NaN in train: {X_train['prev_temp'].isna().sum()}")

In [ ]:
test_with_prev = test_df[test_df["prev_temp"].notna()].copy()
test_no_prev = test_df[test_df["prev_temp"].isna()].copy()

X_test = test_with_prev[feature_cols]
y_test = test_with_prev["true_temp"]

print(f"\nTest set total: {len(test_df)}")
print(f"Test with prev_temp (evaluation set): {len(test_with_prev)}")
print(f"Test without prev_temp: {len(test_no_prev)}")

 ### Dataset structure after scoping

 The filtered dataset contains 12683 rows — 9784 for training and 2899 for
 testing. First measurements have been removed in Notebook 02, restricting the
 model to the prediction problem that matches real LF operations: estimating
 intermediate and final temperatures given a prior reading.

 The train set has zero NaN values in `prev_temp` — every training row has its
 thermal anchor available, which is the correct operational condition. The model
 will never train on rows where it must guess the thermal state from scratch.

 The test set splits into two groups: 738 rows where `prev_temp` is available
 (the evaluation set) and 2161 rows where it is NaN. The NaN rows are a
 competition artifact — they arise when consecutive temperature values were
 masked in `data_temp.csv`, causing NaN to propagate forward through the
 `prev_temp` chain. In real LF operations, these NaN values would not exist:
 the operator always has the previous temperature reading before requesting
 a prediction. The 2161 rows are excluded from final evaluation with this
 explicit justification. The model is evaluated exclusively on the 738 rows
 that represent the genuine operational scenario.


  Why GroupKFold by heat

 Random row-level splitting would be invalid here for two reasons:

 1. **Sequential correlation** — consecutive measurements within a heat are
    not independent. A model trained on measurement 3 of heat X and asked to
    predict measurement 4 of the same heat has seen nearly identical feature
    values and a target that is typically within 10–20°C of its prediction
    target. This inflates validation metrics beyond what the model can deliver
    on truly unseen heats.

 2. **prev_temp leakage** — the previous temperature feature directly
    encodes a target value from the same heat. If measurements from the same
    heat appear in both train and validation, the model effectively sees
    near-copies of validation targets during training.

 GroupKFold ensures that all measurements from a given heat are in the same
 fold — the model must generalize across melts, not within them.

In [ ]:
gkf = GroupKFold(n_splits=5)

for fold, (train_idx, val_idx) in enumerate(gkf.split(X_train, y_train, groups)):
    train_keys = groups.iloc[train_idx].nunique()
    val_keys = groups.iloc[val_idx].nunique()
    print(f"Fold {fold + 1}: train {len(train_idx)} rows ({train_keys} heats) | "
          f"val {len(val_idx)} rows ({val_keys} heats)")

### Split structure

The 5-fold GroupKFold produces balanced folds with approximately 495 heats 
(1957 rows) in validation and 1980 heats (7827 rows) in training per fold. The group constraint guarantees that no heat appears in both train and 
validation within any fold. This means the model is always evaluated on 
heats it has never seen.

### Baseline model

In [ ]:
global_mean = y_train.mean()
print(f"Global mean temperature: {global_mean:.1f} °C")

In [ ]:
baseline_maes = []
baseline_rmses = []

for fold, (train_idx, val_idx) in enumerate(gkf.split(X_train, y_train, groups)):
    val_X = X_train.iloc[val_idx]
    val_y = y_train.iloc[val_idx]

    baseline_pred = val_X["prev_temp"]

    mae = mean_absolute_error(val_y, baseline_pred)
    rmse = root_mean_squared_error(val_y, baseline_pred)
    baseline_maes.append(mae)
    baseline_rmses.append(rmse)
    print(f"Fold {fold + 1}: MAE = {mae:.2f} °C  |  RMSE = {rmse:.2f} °C")

print(f"\nBaseline mean MAE:  {np.mean(baseline_maes):.2f} ± {np.std(baseline_maes):.2f} °C")
print(f"Baseline mean RMSE: {np.mean(baseline_rmses):.2f} ± {np.std(baseline_rmses):.2f} °C")

### Baseline performance

The "no change" baseline achieves a mean MAE of 8.46 ± 0.22°C and RMSE of
12.04 ± 0.41°C across 5 folds. The low variance across folds confirms that
GroupKFold is producing representative splits — no fold contains a
disproportionate share of easy or difficult heats.

The baseline here is physically motivated: since every row has a prior
temperature reading available, predicting "no change since the last measurement"
is the strongest possible naive predictor. An 8.46°C average error from this
assumption is operationally significant — LF tapping temperature windows are
typically ±15–20°C, meaning the baseline already accounts for roughly half
the allowable deviation using nothing more than the previous reading.

The RMSE (12.04°C) exceeds the MAE by ~42%, indicating the presence of
large residuals concentrated on measurements following significant thermal
events — bulk additions, extended heating sessions, or long gaps between
measurements where the "no change" assumption breaks down most severely.
This sets a meaningful bar: XGBoost must learn the cumulative effects of
heating, additions, and elapsed time to improve substantially beyond what
a naive thermal persistence model already provides.

### XGBoost - default parameters

In [ ]:
default_params = {
    "n_estimators": 500,
    "max_depth": 6,
    "learning_rate": 0.1,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "random_state": 42,
    "n_jobs": -1,
}

print("Default parameters:")
for k, v in default_params.items():
    print(f"  {k}: {v}")

In [ ]:
default_maes = []
default_rmses = []

for fold, (train_idx, val_idx) in enumerate(gkf.split(X_train, y_train, groups)):
    fold_X_train = X_train.iloc[train_idx]
    fold_y_train = y_train.iloc[train_idx]
    fold_X_val = X_train.iloc[val_idx]
    fold_y_val = y_train.iloc[val_idx]

    model = xgb.XGBRegressor(**default_params)
    model.fit(fold_X_train, fold_y_train,
              eval_set=[(fold_X_val, fold_y_val)], verbose=False)

    preds = model.predict(fold_X_val)
    mae = mean_absolute_error(fold_y_val, preds)
    rmse = root_mean_squared_error(fold_y_val, preds)
    default_maes.append(mae)
    default_rmses.append(rmse)
    print(f"Fold {fold + 1}: MAE = {mae:.2f} °C  |  RMSE = {rmse:.2f} °C")

print(f"\nDefault XGBoost mean MAE:  {np.mean(default_maes):.2f} ± {np.std(default_maes):.2f} °C")
print(f"Default XGBoost mean RMSE: {np.mean(default_rmses):.2f} ± {np.std(default_rmses):.2f} °C")

In [ ]:
improvement_mae = np.mean(baseline_maes) - np.mean(default_maes)
improvement_pct = improvement_mae / np.mean(baseline_maes) * 100
print(f"MAE improvement over baseline: {improvement_mae:.2f} °C ({improvement_pct:.1f}%)")

### Default XGBoost performance

The default XGBoost model achieves a mean MAE of 6.22 ± 0.12°C and RMSE of
8.74 ± 0.30°C — a 2.24°C (26.5%) improvement in MAE over the baseline.

The improvement is meaningful: XGBoost learned to correct for the perturbations
that make temperature change between consecutive readings — energy delivered
by arc heating sessions, thermal losses from bulk and wire additions, elapsed
time since last heating, and process stage effects captured by the measurement
sequence index. These are exactly the physical drivers that the "no change"
assumption ignores.

The RMSE improvement is proportionally larger than the MAE improvement,
indicating the model is particularly effective at correcting large errors —
cases where the baseline breaks down most severely, such as measurements
following significant bulk additions or long heating gaps. These are
operationally the most important predictions: a large error after a thermal
event could lead to incorrect heating decisions and missed tapping temperature
targets.

Fold variance is low (±0.12°C MAE), confirming that the GroupKFold
splits are representative and the model generalizes consistently across
different subsets of heats.

### Hyperparameter tuning

In [ ]:
def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 300, 1500),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 20),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 10.0, log=True),
        "random_state": 42,
        "n_jobs": -1,
    }

    fold_maes = []
    for train_idx, val_idx in gkf.split(X_train, y_train, groups):
        fold_X_train = X_train.iloc[train_idx]
        fold_y_train = y_train.iloc[train_idx]
        fold_X_val = X_train.iloc[val_idx]
        fold_y_val = y_train.iloc[val_idx]

        model = xgb.XGBRegressor(**params)
        model.fit(fold_X_train, fold_y_train,
                  eval_set=[(fold_X_val, fold_y_val)], verbose=False)

        preds = model.predict(fold_X_val)
        fold_maes.append(mean_absolute_error(fold_y_val, preds))

    return np.mean(fold_maes)

In [ ]:
sampler = optuna.samplers.TPESampler(seed=42)
study = optuna.create_study(direction="minimize", sampler=sampler)
study.optimize(objective, n_trials=50, show_progress_bar=True)

print(f"Best MAE: {study.best_value:.2f} °C")
print(f"\nBest parameters:")
for k, v in study.best_params.items():
    print(f"  {k}: {v}")

In [ ]:
best_params = study.best_params
best_params["random_state"] = 42
best_params["n_jobs"] = -1

tuned_maes = []
tuned_rmses = []
tuned_models = []

for fold, (train_idx, val_idx) in enumerate(gkf.split(X_train, y_train, groups)):
    fold_X_train = X_train.iloc[train_idx]
    fold_y_train = y_train.iloc[train_idx]
    fold_X_val = X_train.iloc[val_idx]
    fold_y_val = y_train.iloc[val_idx]

    model = xgb.XGBRegressor(**best_params)
    model.fit(fold_X_train, fold_y_train,
              eval_set=[(fold_X_val, fold_y_val)], verbose=False)

    preds = model.predict(fold_X_val)
    mae = mean_absolute_error(fold_y_val, preds)
    rmse = root_mean_squared_error(fold_y_val, preds)
    tuned_maes.append(mae)
    tuned_rmses.append(rmse)
    tuned_models.append(model)
    print(f"Fold {fold + 1}: MAE = {mae:.2f} °C  |  RMSE = {rmse:.2f} °C")

print(f"\nTuned XGBoost mean MAE:  {np.mean(tuned_maes):.2f} ± {np.std(tuned_maes):.2f} °C")
print(f"Tuned XGBoost mean RMSE: {np.mean(tuned_rmses):.2f} ± {np.std(tuned_rmses):.2f} °C")

In [ ]:
print("Performance comparison:")
print(f"  Baseline:        MAE = {np.mean(baseline_maes):.2f} ± {np.std(baseline_maes):.2f} °C  |  RMSE = {np.mean(baseline_rmses):.2f} ± {np.std(baseline_rmses):.2f} °C")
print(f"  Default XGBoost: MAE = {np.mean(default_maes):.2f} ± {np.std(default_maes):.2f} °C  |  RMSE = {np.mean(default_rmses):.2f} ± {np.std(default_rmses):.2f} °C")
print(f"  Tuned XGBoost:   MAE = {np.mean(tuned_maes):.2f} ± {np.std(tuned_maes):.2f} °C  |  RMSE = {np.mean(tuned_rmses):.2f} ± {np.std(tuned_rmses):.2f} °C")

improvement_vs_baseline = np.mean(baseline_maes) - np.mean(tuned_maes)
improvement_vs_default = np.mean(default_maes) - np.mean(tuned_maes)
print(f"\n  Improvement vs baseline: {improvement_vs_baseline:.2f} °C ({improvement_vs_baseline / np.mean(baseline_maes) * 100:.1f}%)")
print(f"  Improvement vs default:  {improvement_vs_default:.2f} °C ({improvement_vs_default / np.mean(default_maes) * 100:.1f}%)")

 ### Hyperparameter tuning results

 The tuned model achieves a mean MAE of 5.71 ± 0.15°C and RMSE of
 8.17 ± 0.38°C — a 32.4% improvement over the baseline and 8.1% over the
 default XGBoost configuration.

 | Model | MAE (°C) | RMSE (°C) |
 |---|---|---|
 | Baseline (prev_temp) | 8.46 ± 0.22 | 12.04 ± 0.41 |
 | XGBoost default | 6.22 ± 0.12 | 8.74 ± 0.30 |
 | XGBoost tuned | 5.71 ± 0.15 | 8.17 ± 0.38 |

 The tuned parameters reveal how the model adapts to the scoped dataset — one
 where every row has `prev_temp` available and the prediction problem is
 exclusively incremental:

 - **Low learning rate (0.012) with ~1400 trees** — the model builds its
   predictions through many small corrections. With first measurements removed,
   the target variance is lower (all predictions start from a known thermal
   anchor), so the optimizer favors a more gradual, conservative ensemble that
   avoids chasing noise in the residuals between consecutive readings.

 - **Weak L1 regularization (reg_alpha = 0.015)** — a dramatic shift from the
   original tuning (1.60). With first measurements removed, the model no longer
   needs aggressive sparsity to suppress uninformative features in the
   anchor-less regime. The rare material columns still contribute little, but
   the model can afford to keep small contributions from more features without
   overfitting.

 - **Moderate L2 regularization (reg_lambda = 0.12)** — up from near-zero
   (0.004) in the original tuning. L2 penalizes large weights uniformly,
   and the increased value suggests the model benefits from dampening the
   dominant `prev_temp` feature slightly to let the secondary features
   (energy, time, additions) contribute more to the residual correction.
   This is a more balanced ensemble.

 - **max_depth = 4, min_child_weight = 14** — shallower trees with larger
   minimum leaf size compared to the original (depth 5, min_child_weight 4).
   The scoped dataset is more homogeneous — all rows share the same prediction
   regime (incremental from a known anchor) — so the model needs less complexity
   to capture the relevant patterns. The higher min_child_weight prevents
   splits on small subgroups, further reducing overfitting risk.

 - **Lower subsample (0.66)** — each tree sees only 66% of training rows,
   introducing more diversity across the 1,421 trees. Combined with the low
   learning rate, this produces a highly regularized ensemble that generalizes
   well across different heat conditions.


### Residual analysis

Collecting residuals from all validation folds

In [ ]:
all_val_indices = []
all_residuals = []
all_preds = []
all_actuals = []

for fold, (train_idx, val_idx) in enumerate(gkf.split(X_train, y_train, groups)):
    fold_X_val = X_train.iloc[val_idx]
    fold_y_val = y_train.iloc[val_idx]

    model = tuned_models[fold]
    preds = model.predict(fold_X_val)
    residuals = fold_y_val.values - preds

    all_val_indices.extend(val_idx)
    all_residuals.extend(residuals)
    all_preds.extend(preds)
    all_actuals.extend(fold_y_val.values)

residual_df = pd.DataFrame({
    "actual": all_actuals,
    "predicted": all_preds,
    "residual": all_residuals,
    "measurement_index": X_train.iloc[all_val_indices]["measurement_index"].values,
    "cumulative_energy_MJ": X_train.iloc[all_val_indices]["cumulative_energy_MJ"].values,
    "total_bulk_cum": X_train.iloc[all_val_indices]["total_bulk_cum"].values,
})

print(f"Residual stats:")
print(f"  Mean:   {residual_df['residual'].mean():.2f} °C")
print(f"  Std:    {residual_df['residual'].std():.2f} °C")
print(f"  Median: {residual_df['residual'].median():.2f} °C")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].scatter(residual_df["predicted"], residual_df["residual"], alpha=0.15, s=10)
axes[0].axhline(0, color="red", linewidth=1)
axes[0].set_xlabel("Predicted Temperature (°C)")
axes[0].set_ylabel("Residual (°C)")
axes[0].set_title("Residuals vs Predicted")

residual_df["residual"].hist(bins=60, ax=axes[1], edgecolor="black", alpha=0.7)
axes[1].axvline(0, color="red", linewidth=1)
axes[1].set_xlabel("Residual (°C)")
axes[1].set_ylabel("Count")
axes[1].set_title("Residual Distribution")

axes[2].scatter(residual_df["actual"], residual_df["predicted"], alpha=0.15, s=10)
lims = [residual_df["actual"].min() - 10, residual_df["actual"].max() + 10]
axes[2].plot(lims, lims, color="red", linewidth=1)
axes[2].set_xlabel("Actual Temperature (°C)")
axes[2].set_ylabel("Predicted Temperature (°C)")
axes[2].set_title("Actual vs Predicted")

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

residual_by_idx = residual_df.groupby("measurement_index")["residual"].agg(["mean", "std", "count"])
residual_by_idx = residual_by_idx[residual_by_idx["count"] >= 20]

axes[0].bar(residual_by_idx.index, residual_by_idx["mean"],
            yerr=residual_by_idx["std"], alpha=0.7, capsize=3, edgecolor="black")
axes[0].axhline(0, color="red", linewidth=1)
axes[0].set_xlabel("Measurement Index")
axes[0].set_ylabel("Mean Residual (°C)")
axes[0].set_title("Residuals by Measurement Position")

mae_by_idx = residual_df.groupby("measurement_index")["residual"].apply(
    lambda x: np.mean(np.abs(x))
)
mae_by_idx = mae_by_idx[residual_by_idx.index]

axes[1].bar(mae_by_idx.index, mae_by_idx.values, alpha=0.7, edgecolor="black")
axes[1].set_xlabel("Measurement Index")
axes[1].set_ylabel("MAE (°C)")
axes[1].set_title("MAE by Measurement Position")

plt.tight_layout()
plt.show()

 ### Residual patterns

 **Global bias** — the mean residual is −0.02°C with a median of −0.01°C,
 confirming zero systematic bias. The standard deviation of 8.18°C is
 consistent with the cross-validated RMSE of 8.17°C and represents a 41%
 reduction from the original notebook's 13.92°C — a direct consequence of
 removing the high-error first measurements that inflated residual variance.

 **Residual distribution** — approximately symmetric and centered on zero,
 with heavier tails than a normal distribution. The tails reflect the
 irreducible noise in LF temperature measurement: thermocouple immersion
 effects, bath inhomogeneity, and thermal events between consecutive readings
 that the cumulative features cannot fully resolve.

 **Actual vs predicted** — substantially tighter adherence to the diagonal
 than the original notebook. The horizontal band of predictions near 1580°C
 that dominated the previous plot — caused by anchor-less first measurements
 defaulting toward the global mean — is eliminated. The regression-toward-the-
 mean effect persists at the extremes (above 1650°C and below 1550°C) but is
 less pronounced, because the model now has a thermal anchor for every
 prediction and can track departures from the operating center more accurately.

 **MAE by measurement index** — the pattern is now a U-shape rather than the
 steep decline from index 1 seen previously:

 - Index 2 shows the highest MAE (~7°C). This is the second measurement, where
   `prev_temp` is the arrival temperature. The gap between arrival and the
   second reading is typically the largest thermal jump in the heat — the
   initial heating phase delivers substantial energy to bring the bath from
   arrival temperature (~1550–1570°C) toward the operating window (~1590–1610°C).
   This large increment is the hardest to predict incrementally.

 - Indices 4–7 are the model's sweet spot (~4.5–5°C MAE). The heat is in its
   stable middle phase: small temperature adjustments between readings, moderate
   cumulative energy, and well-characterized thermal dynamics. This is where the
   model delivers operationally useful precision — within the thermocouple
   measurement uncertainty of ±3–5°C.

 - Indices 9+ show rising MAE (~6–7°C). These are late-stage measurements in
   complex heats that required extended treatment — tight chemistry specs,
   multiple correction cycles, or thermal recovery after large additions. Only
   ~72 heats reach index 10+, so the model has limited training data in this
   regime and higher variance is expected.

 The U-shape is a physically meaningful signature of the LF process lifecycle,
 not a model deficiency. The model performs best during the stable middle phase
 and struggles at the transitions — early heating ramp-up and late-stage
 operational complexity.

### Final model - test set evaluation

In [ ]:
final_model = xgb.XGBRegressor(**best_params)
final_model.fit(X_train, y_train, verbose=False)

test_preds = final_model.predict(X_test)
test_mae = mean_absolute_error(y_test, test_preds)
test_rmse = root_mean_squared_error(y_test, test_preds)

print(f"Test set (operational scope): {len(X_test)} rows")
print(f"Test MAE:  {test_mae:.2f} °C")
print(f"Test RMSE: {test_rmse:.2f} °C")

print(f"\nComparison:")
print(f"  CV MAE:   {np.mean(tuned_maes):.2f} ± {np.std(tuned_maes):.2f} °C")
print(f"  Test MAE: {test_mae:.2f} °C")

Test set residual analysis

In [ ]:
test_residuals = y_test.values - test_preds

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].scatter(test_preds, test_residuals, alpha=0.3, s=10)
axes[0].axhline(0, color="red", linewidth=1)
axes[0].set_xlabel("Predicted Temperature (°C)")
axes[0].set_ylabel("Residual (°C)")
axes[0].set_title("Test Set — Residuals vs Predicted")

(pd.Series(test_residuals)).hist(bins=40, ax=axes[1], edgecolor="black", alpha=0.7)
axes[1].axvline(0, color="red", linewidth=1)
axes[1].set_xlabel("Residual (°C)")
axes[1].set_ylabel("Count")
axes[1].set_title("Test Set — Residual Distribution")

axes[2].scatter(y_test, test_preds, alpha=0.3, s=10)
lims = [y_test.min() - 10, y_test.max() + 10]
axes[2].plot(lims, lims, color="red", linewidth=1)
axes[2].set_xlabel("Actual Temperature (°C)")
axes[2].set_ylabel("Predicted Temperature (°C)")
axes[2].set_title("Test Set — Actual vs Predicted")

plt.tight_layout()
plt.show()

Saving predictions

In [ ]:
test_eval = test_df[test_df["prev_temp"].notna()].copy()
test_eval["predicted_temp"] = test_preds
test_eval["residual"] = test_eval["true_temp"] - test_eval["predicted_temp"]
test_eval.to_csv("../data/processed/test_predictions.csv", index=False)
print(f"Saved to data/processed/test_predictions.csv — {len(test_eval)} rows")


 ### Test set results

 The model achieves a test MAE of 6.56°C and RMSE of 9.38°C on the 738
 operationally valid test rows — rows where `prev_temp` is available, matching
 the real LF operating scenario where the operator always has the prior reading.

 | Metric | CV (5-fold) | Test set |
 |---|---|---|
 | MAE (°C) | 5.71 ± 0.15 | 6.56 |
 | RMSE (°C) | 8.17 ± 0.38 | 9.38 |

 The 0.85°C gap between CV and test MAE is within acceptable bounds and does
 not indicate overfitting. The CV estimate is based on 9,784 rows across all
 operating conditions, while the test set is a 738-row subset selected by the
 competition structure — it may over-represent certain measurement positions
 or melt types. The model generalizes correctly: residuals are centered on zero
 with no systematic bias, and the actual-vs-predicted relationship follows the
 diagonal across the full 1525–1700°C operating range.

 A 6.56°C MAE is operationally meaningful. LF tapping temperature windows are
 typically ±10–15°C — meaning the model's average error is well within half the
 allowable deviation. For the stable mid-heat measurements (indices 4–7), the
 CV analysis showed MAE of 4.5–5°C — approaching the ±3–5°C uncertainty of
 the thermocouple measurement itself. At this level, the model's precision is
 limited not by its learned physics but by the resolution of the instrument
 providing the training labels.


 ## Summary

 This notebook trained an XGBoost regressor to predict steel temperature at
 intermediate and final measurement points within a Ladle Furnace heat, using
 physically motivated features with strict temporal causality. First measurements
 (arrival temperatures) were excluded in Notebook 02 — they are always physically
 measured by the operator and represent a fundamentally different prediction
 problem that depends on upstream variables absent from this dataset.

 **Key results:**

 | Model | CV MAE (°C) | Test MAE (°C) |
 |---|---|---|
 | Baseline (prev_temp) | 8.46 ± 0.22 | — |
 | XGBoost default | 6.22 ± 0.12 | — |
 | XGBoost tuned | 5.71 ± 0.15 | 6.56 |

 **Key findings:**

 1. **Correct scoping transforms the result.** Removing first measurements and
    evaluating only on rows where `prev_temp` is available — the real operational
    scenario — reduces the test MAE from 15.53°C (original notebook) to 6.56°C.
    The original metric was inflated by 2161 test rows where consecutive masking
    in the competition structure propagated NaN through `prev_temp`, a scenario
    that does not occur in real LF operations.

 2. **The model achieves operationally useful precision.** A test MAE of 6.56°C
    is well within half the typical LF tapping temperature window of ±10–15°C.
    During the stable mid-heat phase (measurement indices 4–7), CV MAE drops to
    4.5–5°C — approaching the ±3–5°C uncertainty of the thermocouple itself. At
    this level, model precision is limited by instrument resolution, not by the
    learned physics.

 3. **The tuned model learns meaningful features.** The 32.4% improvement
    over baseline comes from integrating cumulative energy input, time-based heat
    loss, and material addition effects — physical mechanisms that the simple
    "no change" baseline ignores. The tuned parameters shifted significantly from
    the original notebook: shallower trees (depth 4 vs 5), higher min_child_weight
    (14 vs 4), and weaker L1 regularization (0.015 vs 1.60) — reflecting a more
    homogeneous dataset where every row shares the same prediction regime.

 4. **Second measurements are the hardest to predict.** With first measurements
    removed, the U-shaped MAE profile across measurement indices reveals that
    index 2 (~7°C MAE) is the most challenging — the initial heating phase
    produces
